# CelebA: p = 5, 10

One of three **standalone parallel workers**. Use a separate Colab runtime for each notebook; three tabs attached to one runtime are not parallel GPU sessions. Run **CelebA_P50** first so its preparation cell freezes all five shared deployments, then run the other two. All notebooks can then run concurrently if Colab grants the sessions. This job also runs the private-data-noise comparator at p=10. It owns 630 utility fits in total and the single-writer MP/AM/OP replay and combined reporting stage.

## Preserved scientific protocol

- Five seeds (0–4); 10 positive anchor-noise SDs 0.006, 0.012, …, 0.060; four independent alternative releases at each level: **200 NAA fits per p**. Zero is evaluated once through noiseless C-GDP, I-GDP and AA-GDP controls. The complete experiment has **1,525 fitted models**, not 1,525 additional fits after migration.
- Every seed has one 50-participant deployment; p=2,5,10,20,50 are nested prefixes of that same deployment. Public uncentered SVD: 10,000 images, h=400; r=1,600 anchors; one colluder/reference participant 0 and target participant 1.
- Each participant has 100 identities: 70 training, 15 validation, 15 test identities. One reference image per identity is withheld from all uploads. All remaining images of an identity stay together in its utility split. Identity and image counts are recorded separately.
- Smiling utility: MLP 400→256→128→1, ReLU, Adam lr=0.001, batch 256, maximum 100 epochs, patience 10, best validation balanced accuracy. The training seed stays fixed within a deployment. Report pooled test balanced accuracy. Local models use only their own participant data and pool exact test confusion counts across prefixes.
- Float64 GPM alignment, spectral initialization, absolute step tolerance 1e-7, maximum 500 iterations. Retain and identify capped runs.
- C-GDP private-data noise is evaluated at p=10, entrywise SD 0.3,0.6,…,3.0, with the exact noise from the uploaded query record.
- MP, AM and OP are the original reconstructed-image attacks. The frozen VGGFace2 InceptionResnetV1 auditor matches a reconstructed query against a different clean image of the same identity and nine decoys. 100 target identities × 10 fixed ten-candidate lineups = 1,000 trials; chance is 10%. The 100-reference pool stays fixed across p. MP/OP depend on the target anchors; AM additionally uses the p-specific alignment and the colluder's known rotation. I-GDP and Local-only privacy is unmeasured, not zero.
- Average four releases within each seed/level, then report mean and SD across five seed means. This is evidence for the specified evaluation pipeline, not a general privacy guarantee.

## Checkpoints and concurrency

Completed legacy fits under `results/celeba` are verified and reused without retraining or modification. The p=50 preparation stage reuses the exact existing seed-0 deployment and creates the other four once; workers verify and read these frozen artifacts. Each worker writes only `results/celeba_parallel/jobs/p5_10`. No worker appends to another worker's CSV or overwrites another fit. A lock prevents accidentally launching the same job twice.

The p=5,10 job owns attack replay and merged reports under `results/celeba_parallel/merged`. New runs save the AM reconstruction while alignment is in memory; legacy runs require alignment-only replay for AM, with no utility retraining. If peers are still running, reports remain explicitly incomplete. Rerun the final **Replay and merge** cell after the other jobs finish to complete all attack summaries.

Colab interruptions lose only the unfinished calculation. After a disconnected runtime, confirm it stopped before clearing its stale worker lock; never launch two instances of one job. All implementation is embedded below.


In [ ]:
# Install only the auditor package; keep Colab's matching torch/torchvision builds.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import subprocess
import sys
import importlib.metadata
try:
    installed = importlib.metadata.version("facenet-pytorch")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "2.6.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "facenet-pytorch==2.6.0"])
from google.colab import drive
drive.mount("/content/drive")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Use Runtime > Change runtime type and select a GPU, then Run all.")
print("Connected GPU:", torch.cuda.get_device_name())


In [ ]:
CONFIG = {'seeds': [0, 1, 2, 3, 4], 'realizations': 4, 'anchor_scales': [0.006, 0.012, 0.018, 0.024, 0.03, 0.036, 0.042, 0.048, 0.054, 0.06], 'participant_counts': [2, 5, 10, 20, 50], 'private_sigmas': [0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.1, 2.4, 2.7, 3.0], 'h': 400, 'r': 1600, 'target_participant': 1, 'reference_participant': 0, 'gpm_tolerance': 1e-07, 'gpm_max_iters': 500, 'mlp': {'hidden_dims': [256, 128], 'max_epochs': 100, 'patience': 10, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 0.0}, 'old_cache': '/content/drive/MyDrive/NAA-GDP/CelebA_PublicSVD_identity100_cache_v1', 'output_root': '/content/drive/MyDrive/Noisy Anchor Alignment/results/celeba', 'protocol': 'celeba-op-public-auditor-five-seed-v1', 'gallery_pool': '100 reserved references of target participant, fixed across p', 'noise_units': 'absolute entrywise standard deviation in the reduced representation', 'local_baseline': 'one MLP per participant on its own unperturbed public-SVD features; pooled test confusion counts'}
SCIENTIFIC_CODE_SHA256 = '76345fd2ebe4d59150e02c134c962ff1b0f395dc8494bfbc97fa2ac6b9b23eab'
PARALLEL_CODE_SHA256 = '1a980269c1778978c64462bf76cf00949cd95e29e33d5066107809299e5dec35'
JOB_ID = 'p5_10'
PARALLEL_ROOT = '/content/drive/MyDrive/Noisy Anchor Alignment/results/celeba_parallel'
LEGACY_ROOT = CONFIG["output_root"]
CANONICAL_PATH = PARALLEL_ROOT + "/shared/canonical.json"


## data pipeline
Embedded standalone implementation.


In [ ]:
"""Fresh CelebA deployments; old NAA-GDP assets are read-only raw-data inputs."""
from __future__ import annotations

import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import tempfile
import zipfile

import numpy as np
import pandas as pd
from PIL import Image
import torch

DATA_PREPROCESS = "celeba-center-square-rgb64-v1"
DATA_SCHEMA = "naa-celeba-fresh-20260906-v1"


def _data_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def _data_seed(seed, *parts):
    return int(_data_hash([int(seed), *parts])[:15], 16)


def _data_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def _data_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    os.replace(tmp, path)


def _data_order(seed, label, values):
    return sorted(values, key=lambda item: _data_hash([int(seed), label, item]))


def _data_mirror(source, destination, expected):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and _data_sha(destination) == expected:
        return destination
    tmp = destination.with_suffix(destination.suffix + ".tmp")
    shutil.copyfile(source, tmp)
    if _data_sha(tmp) != expected:
        raise RuntimeError(f"Content checksum mismatch: {source}")
    os.replace(tmp, destination)
    return destination


def _data_read_metadata(archive, identity_path):
    with zipfile.ZipFile(archive) as handle:
        matches = [name for name in handle.namelist() if Path(name).name == "list_attr_celeba.csv"]
        if len(matches) != 1:
            raise RuntimeError("Expected one list_attr_celeba.csv in the source archive")
        attr_bytes = handle.read(matches[0])
    identity_bytes = identity_path.read_bytes()
    if identity_path.suffix.lower() == ".txt":
        identity = pd.read_csv(io.BytesIO(identity_bytes), sep=r"\s+", header=None,
                               names=["image_id", "identity"])
    else:
        identity = pd.read_csv(io.BytesIO(identity_bytes))
        identity.columns = ["image_id", "identity"]
    attr = pd.read_csv(io.BytesIO(attr_bytes))
    attr = attr.rename(columns={attr.columns[0]: "image_id"})
    frame = attr[["image_id", "Smiling"]].merge(identity, on="image_id", validate="one_to_one")
    frame = frame.sort_values("image_id").reset_index(drop=True)
    frame["identity"] = frame.identity.astype(np.int64)
    frame["label"] = (frame.Smiling.astype(int) > 0).astype(np.int64)
    if len(frame) != 202599 or frame.identity.nunique() != 10177 or not frame.image_id.is_unique:
        raise RuntimeError("Source must contain all 202,599 images and 10,177 identities")
    return frame, {"attributes": hashlib.sha256(attr_bytes).hexdigest(),
                   "identities": hashlib.sha256(identity_bytes).hexdigest()}


def _data_build_rgb(old_cache, archive, archive_hash, names, local_rgb):
    """Reuse only verified uint8 pixels; generate any missing pixels from raw JPGs."""
    tmp = local_rgb.with_suffix(".building.npy")
    pixels = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8,
                                     shape=(len(names), 3, 64, 64))
    row = {name: i for i, name in enumerate(names)}
    written = np.zeros(len(names), dtype=bool)
    provenance = []
    for path in sorted((old_cache / "rgb64_shards").glob("*/manifest.json")):
        manifest = json.loads(path.read_text())
        if manifest.get("preprocess") != DATA_PREPROCESS:
            continue
        offset = 0
        for record in manifest["shards"]:
            shard_names = manifest["filenames"][offset:offset + int(record["count"])]
            offset += int(record["count"])
            wanted = [(j, row[name]) for j, name in enumerate(shard_names)
                      if name in row and not written[row[name]]]
            if not wanted:
                continue
            shard = path.parent / record["file"]
            if _data_sha(shard) != record["sha256"]:
                raise RuntimeError(f"Old RGB shard checksum mismatch: {shard}")
            payload = torch.load(shard, map_location="cpu", weights_only=True)
            images = payload["images"]
            if list(payload["filenames"]) != shard_names or images.dtype != torch.uint8:
                raise RuntimeError(f"Old RGB shard metadata mismatch: {shard}")
            if tuple(images.shape) != (len(shard_names), 3, 64, 64):
                raise RuntimeError(f"Old RGB shard shape mismatch: {shard}")
            src, dst = zip(*wanted)
            pixels[list(dst)] = images[list(src)].numpy()
            written[list(dst)] = True
            provenance.append({"path": str(shard), "sha256": record["sha256"]})
        if offset != len(manifest["filenames"]):
            raise RuntimeError(f"Incomplete old RGB manifest: {path}")
    print(f"Raw image cache: reused {int(written.sum()):,} verified RGB images", flush=True)
    if not written.all():
        local_archive = _data_mirror(archive, local_rgb.parent / "raw_source.zip", archive_hash)
        with zipfile.ZipFile(local_archive) as handle:
            members = {Path(name).name: name for name in handle.namelist() if name.lower().endswith(".jpg")}
            for count, index in enumerate(np.flatnonzero(~written), 1):
                with Image.open(io.BytesIO(handle.read(members[names[index]]))) as image:
                    image = image.convert("RGB")
                    w, h = image.size
                    side = min(w, h)
                    left, top = (w - side) // 2, (h - side) // 2
                    image = image.crop((left, top, left + side, top + side)).resize(
                        (64, 64), Image.Resampling.BILINEAR)
                    pixels[index] = np.asarray(image, dtype=np.uint8).transpose(2, 0, 1)
                if count % 20000 == 0:
                    print(f"Raw image cache: converted {count:,} additional images", flush=True)
    pixels.flush()
    del pixels
    os.replace(tmp, local_rgb)
    return provenance


def prepare_source(old_cache: Path, new_cache: Path) -> dict:
    """Read old raw archive/metadata/RGB shards; write only to new_cache/runtime tmp."""
    old_cache, new_cache = Path(old_cache).resolve(), Path(new_cache).resolve()
    if old_cache == new_cache or old_cache in new_cache.parents or new_cache in old_cache.parents:
        raise ValueError("Old read-only cache and new cache must be separate directories")
    archive = old_cache / "raw_archive/celeba_kaggle_mirror.zip"
    archive_manifest = old_cache / "raw_archive/celeba_kaggle_mirror_manifest.json"
    if not archive.is_file() or not archive_manifest.is_file():
        raise FileNotFoundError(f"Missing old raw archive or manifest: {archive}")
    old_info = json.loads(archive_manifest.read_text())
    if archive.stat().st_size != int(old_info["bytes"]):
        raise RuntimeError("Old raw archive size differs from its manifest")
    candidates = [p for pattern in ("identity_CelebA.txt", "identity_CelebA.csv",
                                    "list_identity_celeba.txt", "list_identity_celeba.csv")
                  for p in old_cache.rglob(pattern)]
    if not candidates:
        raise FileNotFoundError("Official CelebA identity metadata is missing from the old raw cache")
    identity_path = sorted(candidates)[0]
    frame, metadata_hashes = _data_read_metadata(archive, identity_path)
    spec = {"schema": DATA_SCHEMA, "preprocess": DATA_PREPROCESS,
            "archive_sha256": old_info["sha256"], "metadata_sha256": metadata_hashes,
            "image_order_sha256": _data_hash(frame.image_id.tolist())}
    source_hash = _data_hash(spec)
    persistent = new_cache / "raw_rgb64" / source_hash[:20]
    persistent.mkdir(parents=True, exist_ok=True)
    local = Path(tempfile.gettempdir()) / "naa_celeba_raw" / source_hash[:20]
    local.mkdir(parents=True, exist_ok=True)
    rgb, info_path = persistent / "images.npy", persistent / "manifest.json"
    local_rgb = local / "images.npy"
    if info_path.is_file() and rgb.is_file():
        info = json.loads(info_path.read_text())
        if info["spec"] != spec:
            raise RuntimeError("Raw RGB cache configuration mismatch")
        _data_mirror(rgb, local_rgb, info["rgb_sha256"])
    else:
        reused = _data_build_rgb(old_cache, archive, old_info["sha256"],
                                 frame.image_id.tolist(), local_rgb)
        checksum = _data_sha(local_rgb)
        _data_mirror(local_rgb, rgb, checksum)
        info = {"spec": spec, "source_hash": source_hash, "rgb_sha256": checksum,
                "readonly_inputs": [{"path": str(archive), "sha256": old_info["sha256"]},
                                    {"path": str(identity_path), "sha256": metadata_hashes["identities"]},
                                    {"path": str(archive_manifest), "sha256": _data_sha(archive_manifest)},
                                    *reused]}
        _data_json(info_path, info)
    pixels = np.load(local_rgb, mmap_mode="r")
    if pixels.shape != (len(frame), 3, 64, 64) or pixels.dtype != np.uint8:
        raise RuntimeError("New raw RGB cache has an invalid shape or dtype")
    return {"frame": frame, "pixels": pixels, "source_hash": source_hash,
            "source_manifest": info, "new_cache": new_cache}


def _data_allocate(frame, seed, max_participants):
    groups = {int(identity): group.index.tolist() for identity, group in frame.groupby("identity")}
    eligible = [identity for identity, rows in groups.items() if len(rows) >= 2]
    public_ids, n = [], 0
    for identity in _data_order(seed, "public-identities", eligible):
        public_ids.append(identity)
        n += len(groups[identity])
        if n >= 10000:
            break
    public_set = set(public_ids)
    public_rows = _data_order(seed, "public-images", [i for identity in public_ids for i in groups[identity]])[:10000]
    chosen = _data_order(seed, "participant-identities", [i for i in eligible if i not in public_set])[:max_participants * 100]
    if len(public_rows) != 10000 or len(chosen) != max_participants * 100:
        raise RuntimeError("Insufficient eligible public/participant identities")
    stats = []
    for identity in chosen:
        rows = _data_order(seed, f"identity-{identity}-images", groups[identity])
        stats.append({"identity": identity, "reference_row": rows[0], "rows": rows[1:],
                      "n": len(rows) - 1, "positive": int(frame.loc[rows[1:], "label"].sum())})
    capacities = {(p, split): cap for p in range(max_participants)
                  for split, cap in (("train", 70), ("val", 15), ("test", 15))}
    buckets, loads, positives = ({k: [] for k in capacities}, {k: 0 for k in capacities}, {k: 0 for k in capacities})
    mean_n = sum(s["n"] for s in stats) / len(stats)
    prevalence = sum(s["positive"] for s in stats) / sum(s["n"] for s in stats)
    stats.sort(key=lambda s: (-s["n"], -abs(s["positive"] / s["n"] - prevalence),
                              _data_hash([seed, "allocation", s["identity"]])))
    for item in stats:
        def score(key):
            target = mean_n * capacities[key]
            # Minimize the CHANGE in the global bucket cost. Comparing absolute
            # post-assignment costs instead would preferentially fill one bucket.
            def cost(n, positive):
                return (n - target) ** 2 / target + .2 * (positive - prevalence * n) ** 2 / max(1, target * prevalence * (1 - prevalence))
            delta = cost(loads[key] + item["n"], positives[key] + item["positive"]) - cost(loads[key], positives[key])
            return delta + .001 * (len(buckets[key]) + 1) / capacities[key], _data_hash([seed, item["identity"], key])
        key = min((k for k in capacities if len(buckets[k]) < capacities[k]), key=score)
        buckets[key].append(item)
        loads[key] += item["n"]
        positives[key] += item["positive"]
    assignments = [{"participant": p, "split": split, **item}
                   for (p, split), items in sorted(buckets.items())
                   for item in sorted(items, key=lambda item: item["identity"])]
    refs = {item["reference_row"] for item in assignments}
    uploads = {i for item in assignments for i in item["rows"]}
    assert len({item["identity"] for item in assignments}) == max_participants * 100
    assert not refs & uploads and not public_set & set(chosen)
    assert all(len(buckets[k]) == cap for k, cap in capacities.items())
    return public_ids, public_rows, assignments


def prepare_deployment(source, seed: int, outdir: Path, h=400, max_participants=50, device=None):
    """Return CPU tensors; refit public SVD and participant splits for every outer seed."""
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    frame, pixels = source["frame"], source["pixels"]
    public_ids, public_rows, assignments = _data_allocate(frame, seed, max_participants)
    spec = {"schema": DATA_SCHEMA, "source_hash": source["source_hash"], "seed": int(seed),
            "h": h, "max_participants": max_participants, "public_images": 10000,
            "uncentered": True, "svd_q": h + 20, "svd_niter": 4,
            "target_participant": 1, "colluder_participant": 0,
            "gallery_pool": "target participant's 100 reserved clean references, fixed across p",
            "public_identity_ids": public_ids, "public_rows": public_rows,
            "assignments": assignments}
    config_hash = _data_hash(spec)
    cache = source["new_cache"] / "deployments" / config_hash[:20]
    cache.mkdir(parents=True, exist_ok=True)
    data_path, manifest_path = cache / "deployment.pt", cache / "manifest.json"
    if data_path.is_file() and manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        if manifest["config_hash"] != config_hash or _data_sha(data_path) != manifest["data_sha256"]:
            raise RuntimeError("Deployment cache failed configuration/content verification")
        result = torch.load(data_path, map_location="cpu", weights_only=True)
        result["manifest"] = manifest
        _data_json(outdir / "deployment_manifest.json", manifest)
        return result
    print(f"Seed {seed}: fitting fresh public SVD on 10,000 images", flush=True)
    matrix = torch.from_numpy(np.array(pixels[public_rows])).flatten(1).to(device, torch.float32).div_(255)
    with torch.random.fork_rng(devices=[device.index or 0] if device.type == "cuda" else []):
        torch.manual_seed(_data_seed(seed, "public-svd"))
        _, singular, candidate = torch.pca_lowrank(matrix, q=h + 20, center=False, niter=4)
    F = torch.linalg.qr(candidate[:, :h].double().cpu(), mode="reduced")[0].float()
    residual = float(torch.linalg.matrix_norm(F.double().T @ F.double() - torch.eye(h)))
    if residual > 1e-3:
        raise RuntimeError(f"Projection orthogonality check failed: {residual}")
    del matrix, candidate
    basis = F.to(device)
    def project(rows):
        blocks = []
        for start in range(0, len(rows), 512):
            batch = torch.from_numpy(np.array(pixels[rows[start:start + 512]])).flatten(1).to(device, torch.float32).div_(255)
            blocks.append((batch @ basis).cpu())
        return torch.cat(blocks)
    participants, samples = [], []
    queries, query_rows, query_ids, query_locations, refs = [], [], [], [], []
    for p in range(max_participants):
        participant = {}
        for split in ("train", "val", "test"):
            items = [a for a in assignments if a["participant"] == p and a["split"] == split]
            rows = [i for a in items for i in a["rows"]]
            q = project(rows)
            participant[f"{split}_q"] = q
            participant[f"{split}_y"] = torch.tensor(frame.loc[rows, "label"].to_numpy(), dtype=torch.float32)
            participant[f"{split}_identities"] = torch.tensor(frame.loc[rows, "identity"].to_numpy())
            samples.append({"participant": p, "split": split, "images": len(rows),
                            "identities": len(items), "smiling": int(frame.loc[rows, "label"].sum())})
            if p == 1:
                lookup = {row: j for j, row in enumerate(rows)}
                for a in items:
                    row = _data_order(seed, f"query-{a['identity']}", a["rows"])[0]
                    queries.append(q[lookup[row]])
                    query_rows.append(row)
                    query_ids.append(a["identity"])
                    query_locations.append({"split": split, "position": lookup[row]})
                    refs.append(a["reference_row"])
        participants.append(participant)
        print(f"Seed {seed}: projected participant {p + 1}/{max_participants}", flush=True)
    rng = np.random.default_rng(_data_seed(seed, "fixed-target-gallery"))
    lineups = np.empty((100, 10, 10), dtype=np.int64)
    correct = np.empty((100, 10), dtype=np.int64)
    for i in range(100):
        for episode in range(10):
            candidates = np.r_[i, rng.choice(np.delete(np.arange(100), i), 9, replace=False)]
            rng.shuffle(candidates)
            lineups[i, episode], correct[i, episode] = candidates, int(np.flatnonzero(candidates == i)[0])
    assert len(queries) == 100 and not set(query_rows) & set(refs)
    result = {"participants": participants, "F": F, "query_q": torch.stack(queries),
              "gallery_images": torch.from_numpy(np.array(pixels[refs])).float().div_(255),
              "query_images": torch.from_numpy(np.array(pixels[query_rows])).float().div_(255),
              "lineup_indices": torch.from_numpy(lineups), "correct_positions": torch.from_numpy(correct),
              "query_locations": query_locations, "query_identities": query_ids}
    tmp = data_path.with_suffix(".tmp")
    torch.save(result, tmp)
    os.replace(tmp, data_path)
    manifest = {"config_hash": config_hash, "spec": spec, "sample_counts": samples,
                "query_rows": query_rows, "reference_rows": refs, "query_identities": query_ids,
                "lineups": lineups.tolist(), "correct_positions": correct.tolist(),
                "projection_orthogonality_residual": residual,
                "projection_sha256": hashlib.sha256(F.numpy().tobytes()).hexdigest(),
                "retained_singular_values": singular[:h].cpu().tolist(),
                "data_sha256": _data_sha(data_path), "data_path": str(data_path),
                "source_manifest": source["source_manifest"]}
    _data_json(manifest_path, manifest)
    _data_json(outdir / "deployment_manifest.json", manifest)
    result["manifest"] = manifest
    return result


## models protocol
Embedded standalone implementation.


In [ ]:
"""Standalone CelebA protocol, MLP, and frozen reconstruction-auditor component.

Records are row vectors. Protocol arithmetic is float64; utility and face models
use float32. Rates returned by this module are fractions, not percentages.
"""

import hashlib
import json
import os
import random
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from torch import nn
from torch.nn import functional as torch_functional


def _model_seed(seed, *parts):
    payload = json.dumps([int(seed), *parts], separators=(",", ":"), ensure_ascii=True)
    return int.from_bytes(hashlib.sha256(payload.encode()).digest()[:4], "little")


def _model_numpy(value, dtype=np.float64):
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=dtype)


def _model_sync(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)


def _model_haar(rng, h):
    q, r = np.linalg.qr(rng.standard_normal((h, h)))
    signs = np.where(np.diag(r) >= 0.0, 1.0, -1.0)
    return q * signs[None, :]


def generate_bank(seed, pmax=50, h=400, r=1600):
    """Stable participant-indexed draws; changing pmax cannot change a prefix.

    A is centered and satisfies A.T @ A = I. O_i are independent Haar draws
    on O(h); translations have independent N(0,1) coordinates.
    """
    if not (1 <= h < r and pmax >= 1):
        raise ValueError("Require 1 <= h < r and pmax >= 1.")
    rng = np.random.default_rng(_model_seed(seed, "anchor"))
    a = rng.standard_normal((r, h))
    a -= a.mean(axis=0)
    a, triangular = np.linalg.qr(a, mode="reduced")
    a *= np.where(np.diag(triangular) >= 0, 1.0, -1.0)[None, :]
    rotations, translations = [], []
    for participant in range(pmax):
        rotations.append(_model_haar(np.random.default_rng(
            _model_seed(seed, "rotation", participant)), h))
        translations.append(np.random.default_rng(
            _model_seed(seed, "translation", participant)).standard_normal(h))
    return {"A": a, "O": np.stack(rotations), "psi": np.stack(translations),
            "seed": int(seed), "h": int(h), "r": int(r), "pmax": int(pmax)}


def noisy_anchor_bank(bank, seed, level_index, realization, v):
    """One alternative release. Noise is iid over entries and participants.

    A level/realization's participant prefix is independent of requested p.
    This does not center W before release: its mean perturbs the translation.
    """
    if not np.isfinite(v) or v < 0:
        raise ValueError("Anchor scale must be finite and nonnegative.")
    if "clean_anchors" not in bank:
        bank["clean_anchors"] = [bank["A"] @ o + psi
                                 for o, psi in zip(bank["O"], bank["psi"])]
    if v == 0:
        return [b.copy() for b in bank["clean_anchors"]]
    return [b + float(v) * np.random.default_rng(_model_seed(
        seed, "anchor-noise", int(level_index), int(realization), participant
    )).standard_normal(b.shape)
        for participant, b in enumerate(bank["clean_anchors"])]


def upload_blocks(blocks, bank):
    """Create private-data uploads without perturbation, Y_i = Q_i O_i + psi_i."""
    return [_model_numpy(q) @ bank["O"][i] + bank["psi"][i]
            for i, q in enumerate(blocks)]


def _model_polar(matrix):
    u, _, vh = torch.linalg.svd(matrix, full_matrices=False)
    return u @ vh


def align_anchors(B, device="cuda", tol=1e-7, max_iters=500, reference=0):
    """Ling spectral initialization and unregularized generalized power method.

    No (p*h)-square Gram is allocated. If r < p*h, spectral initialization
    diagonalizes D D.T and recovers right singular vectors as D.T U / s.
    Iteration is exactly R_next = block_polar(D.T D R), as in the old code.
    Stopping uses the OLD absolute Frobenius step <= tol (not relative step).
    The common output gauge is normalized to the reference participant.

    Returns (maps, diagnostics), with maps['rotations_R'] of shape (p,h,h)
    and maps['means_b'] of shape (p,h). Aligned data are
    (Y_i-b_i) @ maps['rotations_R'][i] + b_reference.
    """
    dev = torch.device(device)
    if not B or max_iters < 1 or tol <= 0 or not 0 <= reference < len(B):
        raise ValueError("Invalid anchors, reference, or solver settings.")
    _model_sync(dev)
    started = time.perf_counter()
    blocks = [torch.as_tensor(b, dtype=torch.float64, device=dev) for b in B]
    r, h = blocks[0].shape
    p = len(blocks)
    if any(tuple(b.shape) != (r, h) for b in blocks) or r <= h:
        raise ValueError("Every anchor block must have common shape r x h, with r > h.")
    means = torch.stack([b.mean(dim=0) for b in blocks])
    D = torch.cat([b - b.mean(dim=0) for b in blocks], dim=1)
    del blocks
    if r < p * h:
        left_gram = D @ D.T
        eigenvalues, eigenvectors = torch.linalg.eigh(left_gram)
        eigenvalues = eigenvalues.flip(0)
        top = eigenvalues[:h]
        if float(top[-1].item()) <= torch.finfo(torch.float64).eps * float(top[0].item()):
            raise RuntimeError("Spectral initialization has a numerically deficient top-h subspace.")
        right = (D.T @ eigenvectors[:, -h:].flip(1)) / top.sqrt().unsqueeze(0)
        gap = float((eigenvalues[h - 1] - eigenvalues[h]).item())
        scale = float(top[-1].item())
        del eigenvectors, eigenvalues, left_gram
    else:
        _, singular, vh = torch.linalg.svd(D, full_matrices=False)
        right = vh[:h].T
        scale = float(singular[h - 1].square().item())
        next_value = float(singular[h].square().item()) if len(singular) > h else 0.0
        gap = scale - next_value
        del vh, singular
    stacked = _model_polar(right.reshape(p, h, h)).reshape(p * h, h)
    del right
    _model_sync(dev)
    initialization_seconds = time.perf_counter() - started
    objective_history = [float((D @ stacked).square().sum().item())]
    step_history = []
    converged = False
    iteration_started = time.perf_counter()
    for _ in range(max_iters):
        next_stacked = _model_polar((D.T @ (D @ stacked)).reshape(p, h, h)).reshape(p*h, h)
        step = float(torch.linalg.vector_norm(next_stacked - stacked).item())
        step_history.append(step)
        stacked = next_stacked
        objective_history.append(float((D @ stacked).square().sum().item()))
        if step <= tol:
            converged = True
            break
    _model_sync(dev)
    iteration_seconds = time.perf_counter() - iteration_started
    next_stacked = _model_polar((D.T @ (D @ stacked)).reshape(p, h, h)).reshape(p*h, h)
    fixed_absolute = float(torch.linalg.vector_norm(next_stacked - stacked).item())
    fixed_relative = fixed_absolute / max(1.0, float(torch.linalg.vector_norm(stacked).item()))
    rotations = stacked.reshape(p, h, h)
    # This exact gauge normalization is also applied to the aligned mean.
    rotations = rotations @ rotations[reference].T.clone()
    rotations[reference] = torch.eye(h, device=dev, dtype=torch.float64)
    template = sum((D[:, i*h:(i+1)*h] @ rotations[i] for i in range(p))) / p
    residual = sum(float((D[:, i*h:(i+1)*h] @ rotations[i] - template).square().sum().item())
                   for i in range(p))
    denominator = float(D.square().sum().item())
    maps = {"rotations_R": rotations.detach().cpu().numpy(),
            "means_b": means.detach().cpu().numpy(), "reference": int(reference)}
    _model_sync(dev)
    diagnostics = {
        "algorithm": "spectral-unregularized-GPM",
        "dtype": "float64", "max_iters": int(max_iters), "tolerance": float(tol),
        "stopping_rule": "absolute Frobenius norm of successive stacked iterates",
        "iterations": len(step_history), "converged": bool(converged),
        "termination_reason": "step_tolerance" if converged else "iteration_cap",
        "last_step": step_history[-1], "fixed_point_residual": fixed_relative,
        "fixed_point_residual_absolute": fixed_absolute,
        "gopp_residual": residual, "relative_gopp_residual": residual / max(denominator, 1e-300),
        "eigengap": gap, "relative_eigengap": gap / max(scale, 1e-300),
        "objective_history": objective_history, "step_history": step_history,
        "initialization_seconds": initialization_seconds, "iteration_seconds": iteration_seconds,
        "elapsed_seconds": time.perf_counter() - started,
        "reference_participant": int(reference), "participant_count": int(p),
    }
    return maps, diagnostics


def apply_maps(uploads, maps):
    means, rotations = maps["means_b"], maps["rotations_R"]
    reference = maps.get("reference", 0)
    return [(_model_numpy(y) - means[i]) @ rotations[i] + means[reference]
            for i, y in enumerate(uploads)]


def op_reconstruct(A, B, Y, device="cuda"):
    """Known-anchor OP recovery of projected records; no image decoder here."""
    dev = torch.device(device)
    a = torch.as_tensor(A, device=dev, dtype=torch.float64)
    b = torch.as_tensor(B, device=dev, dtype=torch.float64)
    y = torch.as_tensor(Y, device=dev, dtype=torch.float64)
    am, bm = a.mean(0), b.mean(0)
    rotation = _model_polar((a - am).T @ (b - bm))
    translation = bm - am @ rotation
    return ((y - translation) @ rotation.T).cpu().numpy()


def _model_set_seed(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def _model_metrics(logits, labels):
    prediction = np.asarray(logits).reshape(-1) >= 0.0
    truth = np.asarray(labels).reshape(-1).astype(bool)
    counts = [int((truth == label).sum()) for label in (False, True)]
    if not all(counts):
        raise ValueError("Balanced accuracy requires both classes in the evaluation split.")
    recall = [float((prediction[truth == label] == label).mean()) for label in (False, True)]
    return {"balanced_accuracy": float(np.mean(recall)),
            "accuracy": float((prediction == truth).mean()),
            "recall_not_smiling": recall[0], "recall_smiling": recall[1],
            "n_records": int(len(truth))}


def fit_mlp(trainX, trainY, valX, valY, testX, testY, seed, cfg=None):
    """Fit the fixed vector MLP and select the epoch using validation BA only.

    seed must be held fixed within a deployment for paired protocol/noise runs.
    A fresh local permutation generator is reset for each call. No feature-wise
    normalization or test-dependent threshold selection is performed.
    """
    config = {"device": "cuda" if torch.cuda.is_available() else "cpu",
              "max_epochs": 50, "patience": 8, "batch_size": 256,
              "lr": 1e-3, "weight_decay": 0.0, "hidden_dims": (256, 128)}
    if cfg is not None:
        config.update(cfg if isinstance(cfg, dict) else vars(cfg))
    if int(config["max_epochs"]) < 1 or int(config["patience"]) < 1:
        raise ValueError("max_epochs and patience must be positive.")
    _model_set_seed(seed)
    dev = torch.device(config["device"])
    arrays = [_model_numpy(x, np.float32) for x in (trainX, valX, testX)]
    labels = [_model_numpy(y, np.float32).reshape(-1) for y in (trainY, valY, testY)]
    h = arrays[0].shape[1]
    if any(x.ndim != 2 or x.shape[1] != h or len(x) != len(y)
           or not np.isfinite(x).all() or not np.isin(y, [0.0, 1.0]).all()
           for x, y in zip(arrays, labels)):
        raise ValueError("MLP requires finite n x h inputs and matching binary 0/1 labels.")
    for y in labels:
        if len(np.unique(y)) != 2:
            raise ValueError("Each utility split must contain both smiling classes.")
    hidden = tuple(config["hidden_dims"])
    if len(hidden) != 2:
        raise ValueError("Expected exactly two MLP hidden layers.")
    network = nn.Sequential(nn.Linear(h, hidden[0]), nn.ReLU(),
                            nn.Linear(hidden[0], hidden[1]), nn.ReLU(),
                            nn.Linear(hidden[1], 1)).to(dev)
    optimizer = torch.optim.Adam(network.parameters(), lr=float(config["lr"]),
                                 weight_decay=float(config["weight_decay"]))
    criterion = nn.BCEWithLogitsLoss()
    xtrain = torch.from_numpy(np.ascontiguousarray(arrays[0]))
    ytrain = torch.from_numpy(labels[0].copy())
    permutation_generator = torch.Generator(device="cpu").manual_seed(int(seed))
    batch_size = int(config["batch_size"])

    def predict(x):
        network.eval()
        output = []
        with torch.no_grad():
            for start in range(0, len(x), batch_size):
                batch = torch.as_tensor(x[start:start+batch_size], device=dev, dtype=torch.float32)
                output.append(network(batch).reshape(-1).cpu().numpy())
        return np.concatenate(output)

    _model_sync(dev)
    started = time.perf_counter()
    best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, int(config["max_epochs"]) + 1):
        network.train()
        permutation = torch.randperm(len(xtrain), generator=permutation_generator)
        loss_sum = 0.0
        for start in range(0, len(xtrain), batch_size):
            indices = permutation[start:start+batch_size]
            xb, yb = xtrain[indices].to(dev), ytrain[indices].to(dev)
            optimizer.zero_grad(set_to_none=True)
            logits = network(xb).reshape(-1)
            loss = criterion(logits, yb)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite MLP training loss.")
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach().item()) * len(indices)
        val_metrics = _model_metrics(predict(arrays[1]), labels[1])
        score = val_metrics["balanced_accuracy"]
        history.append({"epoch": epoch, "train_loss": loss_sum / len(xtrain),
                        "validation_balanced_accuracy": score})
        if score > best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {name: value.detach().cpu().clone()
                          for name, value in network.state_dict().items()}
        else:
            stale += 1
        if stale >= int(config["patience"]):
            break
    network.load_state_dict(best_state)
    # Test records are evaluated exactly once, after the validation selection.
    result = _model_metrics(predict(arrays[2]), labels[2])
    _model_sync(dev)
    result.update({"val_best_balanced_accuracy": float(best_score),
                   "best_epoch": int(best_epoch), "epochs": int(epoch),
                   "train_seconds": time.perf_counter() - started,
                   "training_seed": int(seed), "decision_threshold_logit": 0.0,
                   "architecture": [h, *hidden, 1], "training_history": history,
                   "stopped_early": bool(epoch < int(config["max_epochs"]))})
    del network, optimizer, best_state
    return result


class FaceAuditor:
    """Existing frozen VGGFace2 InceptionResnetV1 cross-image identity auditor."""

    def __init__(self, device="cuda", torchcache=None, batch_size=64):
        if torchcache is not None:
            Path(torchcache).mkdir(parents=True, exist_ok=True)
            os.environ["TORCH_HOME"] = str(torchcache)
        from facenet_pytorch import InceptionResnetV1
        self.device = torch.device(device)
        self.batch_size = int(batch_size)
        self.model = InceptionResnetV1(pretrained="vggface2").eval().to(self.device)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        digest = hashlib.sha256()
        for name, value in sorted(self.model.state_dict().items()):
            digest.update(name.encode())
            digest.update(value.detach().cpu().contiguous().numpy().tobytes())
        self.metadata = {
            "architecture": "InceptionResnetV1", "pretrained": "vggface2",
            "frozen": True, "state_sha256": digest.hexdigest(),
            "preprocessing": "RGB [0,1], bilinear 160x160, (255*x-127.5)/128",
            "comparison": "cosine similarity to clean cross-image references",
        }

    @torch.no_grad()
    def encode_gallery(self, images):
        """Accept uint8 or [0,1] float RGB, NCHW or NHWC, return n x 512 numpy."""
        tensor = torch.as_tensor(images)
        if tensor.ndim != 4:
            raise ValueError("Expected a four-dimensional RGB image batch.")
        if tensor.shape[1] != 3 and tensor.shape[-1] == 3:
            tensor = tensor.permute(0, 3, 1, 2)
        if tensor.shape[1] != 3:
            raise ValueError("Auditor images must have three RGB channels.")
        uint8 = tensor.dtype == torch.uint8
        embeddings = []
        for start in range(0, len(tensor), self.batch_size):
            batch = tensor[start:start+self.batch_size].to(self.device, dtype=torch.float32)
            if uint8:
                batch = batch / 255.0
            if not torch.isfinite(batch).all() or float(batch.min()) < -1e-6 or float(batch.max()) > 1.000001:
                raise ValueError("Auditor floating images must be finite and in [0,1].")
            batch = torch_functional.interpolate(batch, size=(160, 160), mode="bilinear", align_corners=False)
            batch = (batch * 255.0 - 127.5) / 128.0
            embeddings.append(torch_functional.normalize(self.model(batch), dim=1).cpu())
        if not embeddings:
            return np.empty((0, 512), dtype=np.float32)
        return torch.cat(embeddings).numpy()

    @torch.no_grad()
    def linkage(self, qhat, F, gallery_embeddings, lineup_indices, correct_pos):
        """Decode an orthonormal public-SVD representation and score fixed lineups.

        qhat: n_query x h; F: (3*64*64) x h, using CHW flattening.
        lineups: n_query x n_repeated_galleries x n_candidates (or n_query x k).
        Returns identity-balanced linkage when each query represents one identity.
        """
        query = _model_numpy(qhat, np.float32)
        basis = torch.as_tensor(F, dtype=torch.float32, device=self.device)
        if basis.shape[0] != 3 * 64 * 64 or basis.shape[1] != query.shape[1]:
            raise ValueError("F and qhat must use the public RGB64 SVD projection.")
        query_embeddings = []
        for start in range(0, len(query), self.batch_size):
            reduced = torch.as_tensor(query[start:start+self.batch_size], device=self.device)
            decoded = (reduced @ basis.T).clamp(0.0, 1.0).reshape(-1, 3, 64, 64)
            query_embeddings.append(self.encode_gallery(decoded))
        query_embeddings = np.concatenate(query_embeddings)
        gallery = _model_numpy(gallery_embeddings, np.float32)
        indices = np.asarray(lineup_indices, dtype=np.int64)
        truth = np.asarray(correct_pos, dtype=np.int64)
        if indices.ndim == 2:
            indices, truth = indices[:, None, :], truth.reshape(-1, 1)
        if indices.ndim != 3 or len(indices) != len(query) or truth.shape != indices.shape[:2]:
            raise ValueError("Lineup indices and correct positions do not match query count.")
        if (indices < 0).any() or (indices >= len(gallery)).any():
            raise ValueError("Gallery index outside available reference embeddings.")
        if (truth < 0).any() or (truth >= indices.shape[-1]).any():
            raise ValueError("Correct position outside candidate lineup.")
        scores = np.einsum("qd,qgkd->qgk", query_embeddings, gallery[indices])
        success = scores.argmax(axis=-1) == truth
        return {"linkage_rate": float(success.mean()),
                "per_query_success": success.mean(axis=1).tolist(),
                "n_queries": int(len(query)), "lineups_per_query": int(indices.shape[1]),
                "n_candidates": int(indices.shape[2]), "n_lineups": int(success.size),
                "chance_rate": 1.0 / indices.shape[2]}


def protocol_self_test():
    """Small CPU algebra and reference-gauge checks, without pretrained weights."""
    bank = generate_bank(173, pmax=3, h=5, r=17)
    short = generate_bank(173, pmax=2, h=5, r=17)
    np.testing.assert_array_equal(bank["O"][:2], short["O"])
    np.testing.assert_array_equal(bank["A"], short["A"])
    np.testing.assert_allclose(bank["A"].mean(0), 0, atol=2e-15)
    np.testing.assert_allclose(bank["A"].T @ bank["A"], np.eye(5), atol=2e-14)
    data = [np.random.default_rng(i).normal(size=(11, 5)) for i in range(3)]
    uploads = upload_blocks(data, bank)
    clean = noisy_anchor_bank(bank, 173, 0, 0, 0.0)
    maps, diag = align_anchors(clean, device="cpu", tol=1e-10, max_iters=100)
    aligned = apply_maps(uploads, maps)
    for i, result in enumerate(aligned):
        np.testing.assert_allclose(result, data[i] @ bank["O"][0] + bank["psi"][0], atol=1e-11)
        recovered = op_reconstruct(bank["A"], clean[i], uploads[i], device="cpu")
        np.testing.assert_allclose(recovered, data[i], atol=1e-11)
    # Coupling isotropic private noise by orthogonal rotation yields the exact
    # common-reference noisy matrix asserted by the equivalence result.
    errors = [0.1 * np.random.default_rng(500+i).normal(size=q.shape)
              for i, q in enumerate(data)]
    noisy_uploads = [y + errors[i] @ bank["O"][i] for i, y in enumerate(uploads)]
    noisy_aligned = apply_maps(noisy_uploads, maps)
    for i, result in enumerate(noisy_aligned):
        common = (data[i] + errors[i]) @ bank["O"][0] + bank["psi"][0]
        np.testing.assert_allclose(result, common, atol=1e-11)
    np.testing.assert_array_equal(maps["rotations_R"][0], np.eye(5))
    # Common right multiplication must disappear in reference normalization.
    gauge = _model_haar(np.random.default_rng(77), 5)
    raw = maps["rotations_R"] @ gauge
    normalized = raw @ raw[0].T
    np.testing.assert_allclose(normalized, maps["rotations_R"], atol=1e-13)
    noise1 = noisy_anchor_bank(bank, 173, 3, 2, 0.024)
    noise2 = noisy_anchor_bank(short, 173, 3, 2, 0.024)
    for a, b in zip(noise1, noise2):
        np.testing.assert_array_equal(a, b)
    # Exercise the r-square spectral route too (p*h > r).
    wide = generate_bank(31, pmax=5, h=5, r=17)
    _, wide_diag = align_anchors(noisy_anchor_bank(wide, 31, 0, 0, 0), device="cpu",
                                tol=1e-10, max_iters=100)
    assert diag["converged"] and wide_diag["converged"]
    return {"passed": True, "noiseless_gopp_relative_residual": diag["relative_gopp_residual"],
            "wide_spectral_relative_residual": wide_diag["relative_gopp_residual"]}


## reporting helpers
Embedded standalone implementation.


In [ ]:
"""Resumable schedule and seed-level reporting for the standalone notebook."""
import csv
import gc
import importlib.metadata
import platform
import traceback
from datetime import datetime, timezone

SPLITS = ("train", "val", "test")


def _run_completed_counts(root):
    """Count actual fitted models separately from reused local aggregate rows."""
    root = Path(root)
    record_paths = list((root / "records").glob("*.json"))
    aggregate_count = sum("_Local-only_" in path.stem for path in record_paths)
    local_count = sum(1 for _ in (root / "local_models").glob("*.json"))
    return {"completed_fits": len(record_paths) - aggregate_count + local_count,
            "completed_result_rows": len(record_paths),
            "completed_local_fits": local_count,
            "completed_local_aggregates": aggregate_count}


def _run_local_counts(metrics, labels):
    """Recover integer confusion counts from the exact per-class recalls."""
    truth = np.asarray(labels).reshape(-1)
    negative, positive = int((truth == 0).sum()), int((truth == 1).sum())
    if negative + positive != len(truth) or not negative or not positive:
        raise ValueError("Local test labels must contain both binary classes.")
    tn = int(round(metrics["recall_not_smiling"] * negative))
    tp = int(round(metrics["recall_smiling"] * positive))
    if not (0 <= tn <= negative and 0 <= tp <= positive):
        raise RuntimeError("Local confusion counts are inconsistent with test labels.")
    return {"true_negatives": tn, "false_positives": negative - tn,
            "false_negatives": positive - tp, "true_positives": tp}


def _run_pool_local(local_rows):
    """Evaluate all local models' test predictions as one pooled cohort."""
    names = ("true_negatives", "false_positives", "false_negatives", "true_positives")
    counts = {name: sum(row["confusion_counts"][name] for row in local_rows) for name in names}
    tn, fp, fn, tp = (counts[name] for name in names)
    negative, positive = tn + fp, tp + fn
    if negative == 0 or positive == 0:
        raise RuntimeError("Pooled local evaluation requires both classes.")
    return {"balanced_accuracy": 0.5 * (tn / negative + tp / positive),
            "accuracy": (tn + tp) / (negative + positive),
            "recall_not_smiling": tn / negative, "recall_smiling": tp / positive,
            "n_records": negative + positive, "n_local_models": len(local_rows),
            "confusion_counts": counts,
            "aggregation": "pool confusion counts across the participating local models"}


def _run_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n")
    os.replace(temporary, path)


def _run_matmul(x, rotation, shift, device):
    answer = []
    r = torch.as_tensor(rotation, device=device, dtype=torch.float64)
    b = torch.as_tensor(shift, device=device, dtype=torch.float64)
    for start in range(0, len(x), 4096):
        t = torch.as_tensor(x[start:start+4096], device=device, dtype=torch.float64)
        answer.append((t @ r + b).cpu().numpy())
    return np.concatenate(answer)


def _run_collaboration(uploads, maps, p, device):
    result = {}
    for split in SPLITS:
        if maps is None:
            result[split] = np.concatenate(uploads[split][:p]).astype(np.float32)
        else:
            rotations, means = maps["rotations_R"], maps["means_b"]
            reference = maps["reference"]
            result[split] = np.concatenate([
                _run_matmul(uploads[split][i], rotations[i],
                            means[reference] - means[i] @ rotations[i], device).astype(np.float32)
                for i in range(p)])
    return result


def _run_clean_maps(anchors, device):
    b = torch.as_tensor(np.stack(anchors), dtype=torch.float64, device=device)
    means = b.mean(1)
    centered = b - means[:, None, :]
    rotations = _model_polar(centered.transpose(1, 2) @ centered[0])
    rotations[0] = torch.eye(rotations.shape[-1], device=device, dtype=torch.float64)
    return {"rotations_R": rotations.cpu().numpy(), "means_b": means.cpu().numpy(), "reference": 0}


def summarize_results(root, config):
    """First average the four alternative releases, then summarize five deployments."""
    root = Path(root)
    rows = [json.loads(path.read_text()) for path in sorted((root / "records").glob("*.json"))]
    if not rows:
        return pd.DataFrame()
    flat = []
    for row in rows:
        flat.append({key: value for key, value in row.items()
                     if value is None or isinstance(value, (str, int, float, bool))})
    raw = pd.DataFrame(flat)
    raw.to_csv(root / "raw_results.csv", index=False)
    keys = ["method", "participant_count", "noise_scale", "outer_seed"]
    metrics = ["balanced_accuracy", "linkage_rate", "fit_seconds", "alignment_seconds"]
    seed_means = raw.groupby(keys, dropna=False)[metrics].mean().reset_index()
    repetitions = raw.groupby(keys).size().rename("completed_realizations").reset_index()
    seed_means = seed_means.merge(repetitions, on=keys)
    seed_means["required_realizations"] = np.where(seed_means.noise_scale > 0, config["realizations"], 1)
    seed_means["seed_cell_complete"] = seed_means.completed_realizations == seed_means.required_realizations
    seed_means.to_csv(root / "seed_means.csv", index=False)
    complete = seed_means[seed_means.seed_cell_complete]
    summary_rows = []
    for key, group in complete.groupby(keys[:3], dropna=False):
        record = dict(zip(keys[:3], key))
        record.update(n_seeds=len(group), final=len(group) == len(config["seeds"]))
        for metric in metrics:
            values = group[metric].dropna()
            record[metric + "_mean"] = values.mean() if len(values) else np.nan
            record[metric + "_seed_sd"] = values.std(ddof=1) if len(values) > 1 else np.nan
        summary_rows.append(record)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(root / "summary.csv", index=False)
    raw[raw.participant_count == 10].to_csv(root / "main_p10_results.csv", index=False)
    diagnostics = raw[raw.method == "NAA-GDP"]
    cols = [c for c in ("run_id", "outer_seed", "participant_count", "noise_scale", "realization",
                        "gpm_converged", "gpm_iterations", "gpm_last_step", "alignment_seconds") if c in raw]
    diagnostics[cols].to_csv(root / "alignment_diagnostics.csv", index=False)
    counts = _run_completed_counts(root)
    schedule_complete = (len(raw) == expected_result_rows(config)
                         and counts["completed_fits"] == expected_fits(config))
    if len(summary):
        _run_plots(root, summary, schedule_complete)
    noisy = raw[raw.method == "NAA-GDP"].groupby("participant_count").size().to_dict()
    report = ["# CelebA run status", "", f"Completed fitted models: {counts['completed_fits']}/{expected_fits(config)}.",
              f"Completed main result rows: {len(raw)}/{expected_result_rows(config)}.",
              f"Completed individual local fits: {counts['completed_local_fits']}/{len(config['seeds']) * max(config['participant_counts'])}.",
              f"Completed positive-noise NAA fits by participant count: {noisy}.", "",
              "All rates in CSV files are fractions. SD is across outer-seed means, after averaging",
              "four realizations within each seed and noise level. A summary row is final only when",
              "all five seeds have all required realizations. Partial figures are labelled provisional.", "",
              "The main p=10 CSV is an exact filter of raw_results.csv. No rerun or new deployment",
              "is used for the participant-count figure. The same target participant and fixed",
              "100-reference gallery pool are retained across participant counts.", "",
              "Local-only fits one MLP per participant on its own unperturbed public-SVD records.",
              "Each local model is fitted once per seed and reused in nested collaborations.",
              "Local-only result rows pool confusion counts across the first p local models, so",
              "their balanced accuracy uses the same pooled test cohort as collaboration methods.",
              "They are aggregate rows, not additional fits. Their fit_seconds is the sum of the",
              "constituent local fitting times. Privacy is unmeasured, not zero.", "",
              "This evaluates OP reconstruction followed by a frozen public face auditor; it does",
              "not establish privacy against all admissible attacks. I-GDP and Local-only have no linkage estimate."]
    (root / "REPORT.md").write_text("\n".join(report) + "\n")
    return raw


def _run_plots(root, summary, schedule_complete=False):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.2, 4.5))
    for method in ("NAA-GDP", "C-GDP-private-noise"):
        series = summary[(summary.method == method) & (summary.participant_count == 10)].sort_values("noise_scale")
        if len(series):
            ax.errorbar(100*series.linkage_rate_mean, 100*series.balanced_accuracy_mean,
                        xerr=100*series.linkage_rate_seed_sd.fillna(0),
                        yerr=100*series.balanced_accuracy_seed_sd.fillna(0), marker="o", capsize=2, label=method)
    for method in ("C-GDP", "AA-GDP"):
        series = summary[(summary.method == method) & (summary.participant_count == 10)]
        if len(series):
            ax.scatter(100*series.linkage_rate_mean, 100*series.balanced_accuracy_mean, label=method)
    for method, color, style in (("Local-only", "tab:green", "--"), ("I-GDP", "grey", "-.")):
        baseline = summary[(summary.method == method) & (summary.participant_count == 10)]
        if len(baseline):
            ax.axhline(100*baseline.balanced_accuracy_mean.iloc[0], linestyle=style,
                       color=color, label=method + " utility")
    ax.axvline(10, linestyle=":", color="grey", label="10-candidate chance")
    ax.set(xlabel="Frozen-auditor identity linkage (%)", ylabel="Smiling balanced accuracy (%)",
           title="CelebA, p=10" + (" (provisional)" if not schedule_complete else ""))
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(root / "privacy_utility.pdf")
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    naa = summary[summary.method == "NAA-GDP"]
    scales = sorted(naa.noise_scale.unique())
    selected = sorted({scales[index] for index in (0, len(scales)//2, len(scales)-1)}) if scales else []
    for scale in selected:
        series = naa[naa.noise_scale == scale].sort_values("participant_count")
        axes[0].errorbar(series.participant_count, 100*series.balanced_accuracy_mean,
                         yerr=100*series.balanced_accuracy_seed_sd.fillna(0), marker=".", label=f"NAA v={scale:g}")
        axes[1].plot(series.participant_count, series.alignment_seconds_mean, marker=".", label=f"v={scale:g}")
    for method, style in (("Local-only", "--"), ("I-GDP", "-.")):
        series = summary[summary.method == method].sort_values("participant_count")
        if len(series):
            axes[0].errorbar(series.participant_count, 100*series.balanced_accuracy_mean,
                             yerr=100*series.balanced_accuracy_seed_sd.fillna(0),
                             marker="o", linestyle=style, label=method)
    axes[0].set(xlabel="Participant count p", ylabel="Pooled smiling balanced accuracy (%)")
    axes[1].set(xlabel="Participant count p", ylabel="Alignment wall time (s)")
    for ax in axes:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    fig.suptitle("Participant-count sensitivity" + (" (provisional)" if not schedule_complete else ""))
    fig.tight_layout()
    fig.savefig(root / "participant_scaling.pdf")
    plt.close(fig)


def expected_fits(config):
    s, r, p = len(config["seeds"]), config["realizations"], len(config["participant_counts"])
    return s * (p * (len(config["anchor_scales"]) * r + 3)
                + len(config["private_sigmas"]) * r + max(config["participant_counts"]))


def expected_result_rows(config):
    s, r, p = len(config["seeds"]), config["realizations"], len(config["participant_counts"])
    return s * (p * (len(config["anchor_scales"]) * r + 4) + len(config["private_sigmas"]) * r)



## parallel prepare
Embedded standalone implementation.


In [ ]:
"""Single-writer preparation of the shared, immutable CelebA deployments."""

def prepare_parallel_experiment(config, scientific_code_sha256, parallel_root):
    legacy = Path(config['output_root'])
    root = Path(parallel_root)
    shared = root / 'shared'
    shared.mkdir(parents=True, exist_ok=True)
    canonical_path = shared / 'canonical.json'
    identity = {'config': config, 'code_sha256': scientific_code_sha256}
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    protocol = json.loads((legacy / 'protocol.json').read_text())
    if protocol['signature'] != signature:
        raise RuntimeError('Legacy scientific configuration/code differs from this notebook.')
    if canonical_path.exists():
        canonical = json.loads(canonical_path.read_text())
        if canonical['scientific_signature'] != signature:
            raise RuntimeError('Canonical configuration changed.')
        for entry in canonical['deployments'].values():
            if _data_sha(Path(entry['data_path'])) != entry['data_sha256']:
                raise RuntimeError('Canonical deployment checksum changed.')
            if _data_sha(Path(entry['bank_path'])) != entry['bank_sha256']:
                raise RuntimeError('Canonical transform bank checksum changed.')
        print('All five shared deployments already verified.', flush=True)
        return canonical_path
    old_progress = json.loads((legacy / 'progress.json').read_text())
    if old_progress['status'] not in ('interrupted', 'failed', 'complete'):
        raise RuntimeError('Stop the legacy notebook before preparing parallel execution.')
    environment = json.loads((legacy / 'environment.json').read_text())
    current = {'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
               'packages': {name: importlib.metadata.version(name) for name in environment['packages']}}
    for key in current:
        if current[key] != environment[key]:
            raise RuntimeError(f'Preparation environment changed: {key}')
    source = prepare_source(Path(config['old_cache']), legacy / 'cache')
    old_source = json.loads((legacy / 'source_manifest.json').read_text())
    if source['source_manifest'] != old_source:
        raise RuntimeError('Source manifest changed.')
    deployments = {}
    for seed in config['seeds']:
        _run_json(shared / 'preparation_progress.json', {'status': 'preparing', 'seed': seed,
            'ready_seeds': list(deployments), 'updated_utc': datetime.now(timezone.utc).isoformat()})
        deployment = prepare_deployment(source, seed, shared / 'deployments' / f'seed_{seed}',
            h=config['h'], max_participants=max(config['participant_counts']), device='cuda')
        manifest = deployment['manifest']
        entry = {key: manifest[key] for key in ('data_path', 'config_hash', 'projection_sha256', 'data_sha256')}
        bank = generate_bank(seed, max(config['participant_counts']), config['h'], config['r'])
        bank_path = shared / 'deployments' / f'seed_{seed}' / 'transform_bank.npz'
        temporary_bank = bank_path.with_suffix('.tmp.npz')
        np.savez(temporary_bank, **{key: bank[key] for key in ('A', 'O', 'psi')})
        os.replace(temporary_bank, bank_path)
        entry.update(bank_path=str(bank_path), bank_sha256=_data_sha(bank_path))
        del bank
        # Completed fits must keep exactly their original projection and split artifact.
        for row_path in (legacy / 'records').glob(f's{seed}_*.json'):
            row = json.loads(row_path.read_text())
            if row['deployment_artifact'] != {k: entry[k] for k in ('config_hash', 'projection_sha256', 'data_sha256')}:
                raise RuntimeError(f'Legacy deployment mismatch: {row_path.name}')
        deployments[str(seed)] = entry
        print(f'Shared deployment {seed} verified ({len(deployments)}/5).', flush=True)
        del deployment
        gc.collect()
        torch.cuda.empty_cache()
    auditor = json.loads((legacy / 'auditor.json').read_text())
    payload = {'schema': 'celeba-parallel-canonical-v1', 'scientific_config': config,
        'scientific_signature': signature, 'scientific_code_sha256': scientific_code_sha256,
        'source_hash': source['source_hash'], 'source_manifest': source['source_manifest'],
        'environment': environment, 'auditor_sha256': auditor['state_sha256'],
        'torchcache': str(legacy / 'cache' / 'torch_home'), 'deployments': deployments,
        'legacy_root': str(legacy), 'created_utc': datetime.now(timezone.utc).isoformat()}
    _run_json(canonical_path, payload)
    _run_json(shared / 'preparation_progress.json', {'status': 'complete', 'ready_seeds': list(deployments),
        'updated_utc': datetime.now(timezone.utc).isoformat()})
    print('Shared preparation complete. All three workers may now run.', flush=True)
    return canonical_path


## attack helpers
Embedded standalone implementation.


In [ ]:
"""The manuscript's three existing known-anchor reconstruction attacks."""


def attack_reconstruct(A, B, Y, method, device="cuda", maps=None, target=1,
                       colluder_rotation=None):
    """Return reconstructed projected records as float64 NumPy rows.

    B is the target participant's r x h anchor block. AM takes the analyst's
    maps plus participant 0's disclosed orthogonal transform. These formulas
    reproduce the original OP, MP, and alignment-calibration attacks.
    """
    method = method.upper()
    if method == "OP":
        return op_reconstruct(A, B, Y, device=device)
    dev = torch.device(device)
    a = torch.as_tensor(A, dtype=torch.float64, device=dev)
    b = torch.as_tensor(B, dtype=torch.float64, device=dev)
    y = torch.as_tensor(Y, dtype=torch.float64, device=dev)
    amean, bmean = a.mean(0), b.mean(0)
    if method == "MP":
        transform = torch.linalg.pinv(a - amean) @ (b - bmean)
        translation = bmean - amean @ transform
        return ((y - translation) @ torch.linalg.pinv(transform)).cpu().numpy()
    if method != "AM":
        raise ValueError("method must be OP, MP, or AM")
    if maps is None or colluder_rotation is None:
        raise ValueError("AM requires analyst alignment maps and the colluder's rotation")
    rotations = torch.as_tensor(maps["rotations_R"], dtype=torch.float64, device=dev)
    colluder = torch.as_tensor(colluder_rotation, dtype=torch.float64, device=dev)
    common = _model_polar(colluder @ rotations[0])
    transform = common @ rotations[target].T
    translation = bmean - amean @ transform
    return ((y - translation) @ transform.T).cpu().numpy()


def attack_self_test():
    bank = generate_bank(501, pmax=3, h=4, r=13)
    anchors = noisy_anchor_bank(bank, 501, 0, 0, 0)
    q = np.random.default_rng(19).normal(size=(9, 4))
    y = q @ bank["O"][1] + bank["psi"][1]
    maps, _ = align_anchors(anchors, device="cpu", tol=1e-10, max_iters=100)
    errors = {}
    for method in ("OP", "MP", "AM"):
        recovered = attack_reconstruct(bank["A"], anchors[1], y, method, device="cpu",
                                       maps=maps, target=1, colluder_rotation=bank["O"][0])
        errors[method] = float(np.linalg.norm(recovered-q) / np.linalg.norm(q))
        np.testing.assert_allclose(recovered, q, atol=1e-11)
    return {"passed": True, "noiseless_relative_errors": errors}


## parallel runner
Embedded standalone implementation.


In [ ]:
"""Disjoint workers over one immutable scientific deployment and legacy run.

This is a scheduling layer. The original protocol, preprocessing, MLP, noise
streams, and stopping rule are provided by the unchanged scientific modules.
"""

PARALLEL_SCHEMA = "celeba-three-workers-v1"
PARALLEL_JOBS = {"p50": (50,), "p2_20": (2, 20), "p5_10": (5, 10)}


def _parallel_exclusive(directory, name):
    from contextlib import contextmanager
    @contextmanager
    def held():
        directory.mkdir(parents=True, exist_ok=True)
        path=directory/name
        token=hashlib.sha256(os.urandom(32)).hexdigest()
        try:
            with path.open("x") as handle:
                json.dump({"token":token,"started_utc":datetime.now(timezone.utc).isoformat()},handle)
        except FileExistsError:
            raise RuntimeError(f"Another owner operation is running, or its interrupted runtime left a lock: {path}")
        try:
            yield
        finally:
            if path.exists() and json.loads(path.read_text()).get("token")==token:
                path.unlink()
    return held()


def parallel_schedule(config, job_id):
    """Exact, disjoint work ownership; local aggregates are not model fits."""
    if job_id not in PARALLEL_JOBS:
        raise ValueError(f"Unknown parallel job: {job_id}")
    if sorted(config["participant_counts"]) != [2, 5, 10, 20, 50]:
        raise ValueError("Scientific configuration must retain all five participant counts.")
    tasks = []
    def add(seed, method, p=None, level=0, repetition=0, scale=0.0, kind="utility", participant=None):
        name = (f"s{seed}_i{participant}_Local-only" if kind == "local" else
                f"s{seed}_p{p}_{method}_l{level}_r{repetition}")
        tasks.append({"run_id": name, "seed": seed, "method": method, "p": p,
                      "level": level, "realization": repetition, "scale": float(scale),
                      "kind": kind, "participant": participant,
                      "fit_count": 0 if kind == "aggregate" else 1, "owner": job_id})
    for seed in config["seeds"]:
        if job_id == "p50":
            for participant in range(50):
                add(seed, "Local-only", kind="local", participant=participant)
            for p in config["participant_counts"]:
                add(seed, "Local-only", p=p, kind="aggregate")
        for p in PARALLEL_JOBS[job_id]:
            for method in ("C-GDP", "I-GDP", "AA-GDP"):
                add(seed, method, p=p)
        for level, v in enumerate(config["anchor_scales"], 1):
            for repetition in range(config["realizations"]):
                for p in PARALLEL_JOBS[job_id]:
                    add(seed, "NAA-GDP", p, level, repetition, v)
        if job_id == "p5_10":
            for level, sigma in enumerate(config["private_sigmas"], 1):
                for repetition in range(config["realizations"]):
                    add(seed, "C-GDP-private-noise", 10, level, repetition, sigma)
    return tasks


def _parallel_hash_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def _parallel_manifest(manifest_path, config=None):
    path = Path(manifest_path)
    manifest = json.loads(path.read_text())
    required = ("scientific_config", "scientific_signature", "source_hash", "source_manifest",
                "environment", "auditor_sha256", "torchcache", "deployments")
    if any(key not in manifest for key in required):
        raise RuntimeError("Canonical manifest is incomplete; finish precomputation before starting workers.")
    if config is not None and manifest["scientific_config"] != config:
        raise RuntimeError("Worker config differs from the complete original scientific config.")
    if set(manifest["deployments"]) != set(map(str, manifest["scientific_config"]["seeds"])):
        raise RuntimeError("All five frozen deployments must exist before workers start.")
    manifest["_canonical_sha256"] = _parallel_hash_file(path)
    manifest["_canonical_path"] = str(path)
    return manifest


def _parallel_artifact(manifest, seed):
    item = manifest["deployments"][str(seed)]
    return {key: item[key] for key in ("config_hash", "projection_sha256", "data_sha256")}


def _parallel_validate_row(row, task, manifest):
    if row.get("run_id") != task["run_id"]:
        raise RuntimeError(f"Checkpoint name mismatch: {task['run_id']}")
    expected = {"signature": manifest["scientific_signature"], "source_hash": manifest["source_hash"],
                "deployment_artifact": _parallel_artifact(manifest, task["seed"]),
                "auditor_sha256": manifest["auditor_sha256"]}
    for key, value in expected.items():
        if row.get(key) != value:
            raise RuntimeError(f"Incompatible completed checkpoint {task['run_id']}: {key}")
    if "canonical_sha256" in row and row["canonical_sha256"] != manifest["_canonical_sha256"]:
        raise RuntimeError(f"Canonical deployment manifest changed: {task['run_id']}")
    if row.get("outer_seed") != task["seed"] or row.get("method") != task["method"]:
        raise RuntimeError(f"Checkpoint scientific coordinates changed: {task['run_id']}")
    if task["kind"] == "local":
        if row.get("participant") != task["participant"]:
            raise RuntimeError("Local participant mismatch")
    elif row.get("participant_count") != task["p"] or float(row.get("noise_scale", -1)) != task["scale"]:
        raise RuntimeError(f"Checkpoint participant count/noise scale changed: {task['run_id']}")
    return row


def _parallel_checkpoint_path(root, task):
    folder = "local_models" if task["kind"] == "local" else "records"
    return Path(root) / folder / (task["run_id"] + ".json")


def _parallel_collect(tasks, manifest, legacy_root, parallel_root):
    """Validate legacy and new records once; duplicate writes are an error."""
    completed = {}
    for task in tasks:
        old_path = _parallel_checkpoint_path(legacy_root, task)
        new_path = _parallel_checkpoint_path(Path(parallel_root) / "jobs" / task["owner"], task)
        existing = [path for path in (old_path, new_path) if path.is_file()]
        if len(existing) > 1:
            raise RuntimeError(f"Duplicate completed fit/aggregate across legacy and new job: {task['run_id']}")
        if existing:
            path = existing[0]
            row = _parallel_validate_row(json.loads(path.read_text()), task, manifest)
            completed[task["run_id"]] = {"row": row, "path": str(path), "legacy": path == old_path}
    return completed


def _parallel_environment(manifest):
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU runtime is required.")
    current = {"python": platform.python_version(), "torch": torch.__version__,
               "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(),
               "packages": {name: importlib.metadata.version(name) for name in
                            ("numpy", "pandas", "torch", "torchvision", "facenet-pytorch", "Pillow", "matplotlib")}}
    for key in ("python", "torch", "cuda", "packages"):
        if current[key] != manifest["environment"][key]:
            raise RuntimeError(f"Frozen run environment mismatch ({key}); restore original versions.")
    return current


def _parallel_load_deployment(manifest, seed):
    """Never generate or mutate canonical data in a worker."""
    item = manifest["deployments"][str(seed)]
    path = Path(item["data_path"])
    if not path.is_file() or _parallel_hash_file(path) != item["data_sha256"]:
        raise RuntimeError(f"Missing/changed canonical deployment file for seed {seed}")
    data = torch.load(path, map_location="cpu", weights_only=True)
    if len(data["participants"]) != 50:
        raise RuntimeError("Canonical deployment must contain all 50 participants.")
    if hashlib.sha256(data["F"].numpy().tobytes()).hexdigest() != item["projection_sha256"]:
        raise RuntimeError("Canonical public projection mismatch")
    if item.get("bank_path"):
        bank_path = Path(item["bank_path"])
        if _parallel_hash_file(bank_path) != item["bank_sha256"]:
            raise RuntimeError("Canonical transformation bank mismatch")
        with np.load(bank_path, allow_pickle=False) as stored:
            bank = {key: stored[key].copy() for key in ("A", "O", "psi")}
    else:
        # Same pure participant-indexed draw function, only when no bank was frozen.
        cfg = manifest["scientific_config"]
        bank = generate_bank(seed, 50, cfg["h"], cfg["r"])
    return data, bank


def _parallel_context(manifest, seed, auditor, device="cuda", make_uploads=True, limit=50):
    data, bank = _parallel_load_deployment(manifest, seed)
    q = {split: [part[split + "_q"].numpy() for part in data["participants"]] for split in SPLITS}
    labels = {split: [part[split + "_y"].numpy() for part in data["participants"]] for split in SPLITS}
    target = manifest["scientific_config"]["target_participant"]
    query = _run_matmul(data["query_q"].numpy(), bank["O"][target], bank["psi"][target], device)
    ctx = {"data": data, "bank": bank, "q": q, "labels": labels, "query_upload": query,
           "gallery": auditor.encode_gallery(data["gallery_images"]), "seed": seed}
    if make_uploads:
        ctx["uploads"] = {split: [_run_matmul(block, bank["O"][i], bank["psi"][i], device)
                                  for i, block in enumerate(q[split][:limit])] for split in SPLITS}
        ctx["common"] = {split: [_run_matmul(block, bank["O"][0], bank["psi"][0], device)
                                 for block in q[split][:limit]] for split in SPLITS}
    return ctx


def _parallel_privacy(ctx, auditor, qhat):
    data = ctx["data"]
    result = auditor.linkage(qhat, data["F"], ctx["gallery"], data["lineup_indices"], data["correct_positions"])
    truth = data["query_q"].numpy()
    result["projected_recovery_relative_error"] = float(np.linalg.norm(qhat-truth) / np.linalg.norm(truth))
    return result


def _parallel_am(ctx, B, maps, target):
    """Original single-colluder AM in the fixed reference-0 gauge."""
    return attack_reconstruct(ctx["bank"]["A"],B[target],ctx["query_upload"],"AM",
                              maps=maps,target=target,colluder_rotation=ctx["bank"]["O"][0])


def _parallel_mp(ctx, B, target, device="cuda"):
    return attack_reconstruct(ctx["bank"]["A"],B[target],ctx["query_upload"],"MP",device=device)


def _parallel_save_am(path, qhat, task, manifest):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, qhat=np.asarray(qhat, dtype=np.float64),
                        run_id=np.array(task["run_id"]), canonical_sha256=np.array(manifest["_canonical_sha256"]))
    os.replace(temporary, path)


def _parallel_base_row(task, manifest, code_hash):
    return {"run_id": task["run_id"], "signature": manifest["scientific_signature"],
            "source_hash": manifest["source_hash"], "deployment_artifact": _parallel_artifact(manifest, task["seed"]),
            "deployment_hash": manifest["deployments"][str(task["seed"])]["config_hash"],
            "auditor_sha256": manifest["auditor_sha256"], "outer_seed": task["seed"],
            "method": task["method"], "fit_count": task["fit_count"],
            "canonical_sha256": manifest["_canonical_sha256"], "execution_code_sha256": code_hash,
            "scheduler_schema": PARALLEL_SCHEMA, "owner_job": task["owner"],
            "migration_provenance": {"scientific_signature": manifest["scientific_signature"],
                                     "canonical_manifest": manifest["_canonical_path"],
                                     "preserved_scientific_config": True}}


def _parallel_report(directory, completed, tasks, config, complete=False):
    """Each writer owns its own reporting directory; no shared CSV append."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    wanted = {task["run_id"] for task in tasks if task["kind"] != "local"}
    rows = [entry["row"] for name, entry in completed.items() if name in wanted]
    if not rows:
        return pd.DataFrame()
    raw = pd.DataFrame([{k:v for k,v in row.items() if v is None or isinstance(v,(str,int,float,bool))} for row in rows])
    raw.to_csv(directory / "raw_results.csv", index=False)
    raw[raw.participant_count == 10].to_csv(directory / "main_p10_results.csv", index=False)
    keys = ["method", "participant_count", "noise_scale", "outer_seed"]
    metrics = ["balanced_accuracy", "linkage_rate", "fit_seconds", "alignment_seconds"]
    cells = raw.groupby(keys, dropna=False)[metrics].mean().reset_index()
    repetitions = raw.groupby(keys).size().rename("completed_realizations").reset_index()
    cells = cells.merge(repetitions, on=keys)
    cells["required_realizations"] = np.where(cells.noise_scale > 0, config["realizations"], 1)
    cells["seed_cell_complete"] = cells.completed_realizations == cells.required_realizations
    cells.to_csv(directory / "seed_means.csv", index=False)
    summary = []
    for key, group in cells[cells.seed_cell_complete].groupby(keys[:3]):
        row = dict(zip(keys[:3], key))
        row.update(n_seeds=len(group), final=len(group) == len(config["seeds"]))
        for metric in metrics:
            values = group[metric].dropna()
            row[metric+"_mean"] = values.mean() if len(values) else np.nan
            row[metric+"_seed_sd"] = values.std(ddof=1) if len(values)>1 else np.nan
        summary.append(row)
    summary = pd.DataFrame(summary)
    summary.to_csv(directory/"summary.csv", index=False)
    if len(summary):
        _run_plots(directory, summary, complete)
    return raw


def run_parallel_experiment(config, code_hash, job_id, canonical_manifest_path,
                            legacy_root, parallel_root, replay_attacks=True):
    """Run only this job's pending tasks; never alter legacy/canonical files."""
    manifest = _parallel_manifest(canonical_manifest_path, config)
    legacy_root, parallel_root = Path(legacy_root), Path(parallel_root)
    legacy_protocol = json.loads((legacy_root/"protocol.json").read_text())
    if legacy_protocol["signature"] != manifest["scientific_signature"] or legacy_protocol["config"] != config:
        raise RuntimeError("Legacy protocol does not match the canonical scientific specification.")
    environment = _parallel_environment(manifest)
    tasks = parallel_schedule(config, job_id)
    root = parallel_root/"jobs"/job_id
    root.mkdir(parents=True, exist_ok=True)
    lock_path = root/"worker.lock"
    token = hashlib.sha256(os.urandom(32)).hexdigest()
    try:
        with lock_path.open("x") as handle:
            json.dump({"token":token,"started_utc":datetime.now(timezone.utc).isoformat()},handle)
    except FileExistsError:
        raise RuntimeError(f"This job already has a worker lock: {lock_path}. Stop its previous runtime before removing a stale lock.")
    started = time.perf_counter()
    completed = {}
    def status(state, current=None, error=None):
        fits = sum(task["fit_count"] for task in tasks if task["run_id"] in completed)
        _run_json(root/"progress.json", {"status":state,"job_id":job_id,"current":current,"error":error,
                  "completed_fits":fits,"scheduled_fits":sum(t["fit_count"] for t in tasks),
                  "completed_tasks":len(completed),"scheduled_tasks":len(tasks),
                  "reused_legacy_fits":sum(t["fit_count"] for t in tasks if t["run_id"] in completed and completed[t["run_id"]]["legacy"]),
                  "updated_utc":datetime.now(timezone.utc).isoformat(),"session_seconds":time.perf_counter()-started})
        return fits
    def save(task, row):
        path = _parallel_checkpoint_path(root, task)
        if path.exists() or _parallel_checkpoint_path(legacy_root,task).exists():
            raise RuntimeError(f"Refusing to overwrite a completed task: {task['run_id']}")
        row["finished_utc"] = datetime.now(timezone.utc).isoformat()
        _run_json(path,row)
        completed[task["run_id"]] = {"row":row,"path":str(path),"legacy":False}
        count = status("running",task["run_id"])
        print(f"[{job_id} {count}/{sum(t['fit_count'] for t in tasks)}] {task['run_id']}: BA={100*row.get('balanced_accuracy',row.get('utility',{}).get('balanced_accuracy',0)):.2f}%",flush=True)
    try:
        protocol = {"schema":PARALLEL_SCHEMA,"job_id":job_id,"code_sha256":code_hash,
                    "canonical_sha256":manifest["_canonical_sha256"],"scientific_signature":manifest["scientific_signature"]}
        protocol_path = root/"protocol.json"
        if protocol_path.exists() and json.loads(protocol_path.read_text()) != protocol:
            raise RuntimeError("Parallel job code/canonical manifest changed; refusing mixed scheduler checkpoints.")
        _run_json(protocol_path,protocol)
        session_name=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        _run_json(root/"sessions"/(session_name+".json"),environment)
        completed = _parallel_collect(tasks,manifest,legacy_root,parallel_root)
        status("starting")
        auditor = FaceAuditor(device="cuda",torchcache=manifest["torchcache"])
        if auditor.metadata["state_sha256"] != manifest["auditor_sha256"]:
            raise RuntimeError("Frozen auditor weight hash differs from canonical evaluator.")
        for seed in config["seeds"]:
            seed_tasks = [t for t in tasks if t["seed"]==seed and t["run_id"] not in completed]
            if not seed_tasks:
                continue
            status("loading_frozen_deployment",f"seed={seed}")
            limit=max(PARALLEL_JOBS[job_id])
            ctx = _parallel_context(manifest,seed,auditor,limit=limit)
            bank, q, y = ctx["bank"],ctx["q"],ctx["labels"]
            clean = noisy_anchor_bank(bank,seed,0,0,0.0)
            clean_maps = _run_clean_maps(clean[:limit],"cuda")
            target=config["target_participant"]
            zero_privacy = _parallel_privacy(ctx,auditor,op_reconstruct(bank["A"],clean[target],ctx["query_upload"],"cuda"))
            training_seed=_model_seed(seed,"utility-training")
            mlp_cfg={**config["mlp"],"device":"cuda"}
            current_noise=None
            for task in seed_tasks:
                status("running",task["run_id"])
                row=_parallel_base_row(task,manifest,code_hash)
                beginning=time.perf_counter()
                method,p=task["method"],task["p"]
                if task["kind"]=="local":
                    i=task["participant"]
                    metrics=fit_mlp(q["train"][i],y["train"][i],q["val"][i],y["val"][i],q["test"][i],y["test"][i],training_seed,mlp_cfg)
                    row.update(participant=i,utility=metrics,confusion_counts=_run_local_counts(metrics,y["test"][i]),
                               fit_seconds=time.perf_counter()-beginning,training_seed=training_seed,privacy=None,linkage_rate=None)
                    save(task,row)
                    continue
                if task["kind"]=="aggregate":
                    locals_=[completed[f"s{seed}_i{i}_Local-only"]["row"] for i in range(p)]
                    metrics=_run_pool_local(locals_)
                    private,diag=None,None
                    fit_seconds=sum(item["fit_seconds"] for item in locals_)
                    row.update(aggregate_model_count=p,constituent_local_run_ids=[item["run_id"] for item in locals_])
                else:
                    private,diag=None,None
                    if method in ("C-GDP","I-GDP","AA-GDP"):
                        inputs=_run_collaboration(ctx["common"] if method=="C-GDP" else ctx["uploads"],
                                                  clean_maps if method=="AA-GDP" else None,p,"cuda")
                        private=zero_privacy if method!="I-GDP" else None
                        if method=="AA-GDP":
                            reference=np.concatenate(ctx["common"]["test"][:p]).astype(np.float32)
                            error=float(np.linalg.norm(inputs["test"]-reference)/np.linalg.norm(reference))
                            if error>1e-6: raise RuntimeError("Noiseless AA/C equivalence failed")
                            row["noiseless_equivalence_error"]=error
                    elif method=="NAA-GDP":
                        key=(task["level"],task["realization"])
                        if current_noise!=key:
                            B=noisy_anchor_bank(bank,seed,*key,task["scale"])
                            op_privacy=_parallel_privacy(ctx,auditor,op_reconstruct(bank["A"],B[target],ctx["query_upload"],"cuda"))
                            current_noise=key
                        maps,diag=align_anchors(B[:p],device="cuda",tol=config["gpm_tolerance"],max_iters=config["gpm_max_iters"])
                        # Save only AM's 100x400 recovered queries, not full alignment matrices.
                        _parallel_save_am(root/"attack_inputs"/(task["run_id"]+".npz"),_parallel_am(ctx,B,maps,target),task,manifest)
                        inputs=_run_collaboration(ctx["uploads"],maps,p,"cuda")
                        private=op_privacy
                        del maps
                    else:
                        noisy={split:[block+task["scale"]*np.random.default_rng(_model_seed(
                            seed,"private-noise",task["level"],task["realization"],split,i)).standard_normal(block.shape)
                            for i,block in enumerate(ctx["common"][split][:p])] for split in SPLITS}
                        query_y=np.stack([noisy[location["split"]][target][location["position"]]
                                          for location in ctx["data"]["query_locations"]])
                        qhat=_run_matmul(query_y-bank["psi"][0],bank["O"][0].T,np.zeros(config["h"]),"cuda")
                        private=_parallel_privacy(ctx,auditor,qhat)
                        inputs=_run_collaboration(noisy,None,p,"cuda")
                        del noisy
                    labels={split:np.concatenate(y[split][:p]) for split in SPLITS}
                    fit_started=time.perf_counter()
                    metrics=fit_mlp(inputs["train"],labels["train"],inputs["val"],labels["val"],inputs["test"],labels["test"],training_seed,mlp_cfg)
                    fit_seconds=time.perf_counter()-fit_started
                    del inputs
                row.update(participant_count=p,noise_scale=task["scale"],level_index=task["level"],realization=task["realization"],
                           balanced_accuracy=metrics["balanced_accuracy"],accuracy=metrics["accuracy"],utility=metrics,
                           privacy=private,linkage_rate=private["linkage_rate"] if private else None,fit_seconds=fit_seconds,
                           alignment_seconds=diag["elapsed_seconds"] if diag else 0.0,alignment=diag,
                           gpm_converged=diag["converged"] if diag else None,gpm_iterations=diag["iterations"] if diag else None,
                           gpm_last_step=diag["last_step"] if diag else None)
                save(task,row)
            _parallel_report(root,completed,tasks,config,len(completed)==len(tasks))
            del ctx,bank,q,y,clean,clean_maps
            gc.collect(); torch.cuda.empty_cache()
        _parallel_report(root,completed,tasks,config,len(completed)==len(tasks))
        status("utility_complete")
        if job_id=="p5_10":
            if replay_attacks:
                replay_parallel_attacks(canonical_manifest_path,legacy_root,parallel_root,code_hash,auditor=auditor)
            merge_parallel_results(canonical_manifest_path,legacy_root,parallel_root)
        return completed
    except BaseException as exc:
        status("interrupted" if isinstance(exc,KeyboardInterrupt) else "failed",error=str(exc))
        if completed: _parallel_report(root,completed,tasks,config,False)
        raise
    finally:
        if lock_path.exists() and json.loads(lock_path.read_text()).get("token")==token:
            lock_path.unlink()


def replay_parallel_attacks(canonical_manifest_path,legacy_root,parallel_root,code_hash,auditor=None):
    with _parallel_exclusive(Path(parallel_root)/"jobs"/"p5_10","attack_replay.lock"):
        return _parallel_replay_impl(canonical_manifest_path,legacy_root,parallel_root,code_hash,auditor)


def _parallel_replay_impl(canonical_manifest_path,legacy_root,parallel_root,code_hash,auditor=None):
    """Owner p5_10 only: score available OP/MP/AM reconstructions, without MLP fits.

    Future workers' rows are intentionally not awaited on a billed GPU. Rerun
    this cell after peers finish; existing attack checkpoints are reused.
    """
    manifest=_parallel_manifest(canonical_manifest_path)
    config=manifest["scientific_config"]
    root=Path(parallel_root)/"jobs"/"p5_10"
    attacks_dir=root/"attacks"; attacks_dir.mkdir(parents=True,exist_ok=True)
    all_tasks=[task for owner in PARALLEL_JOBS for task in parallel_schedule(config,owner)]
    completed=_parallel_collect(all_tasks,manifest,legacy_root,parallel_root)
    available=[t for t in all_tasks if t["method"]=="NAA-GDP" and t["run_id"] in completed]
    if auditor is None:
        _parallel_environment(manifest)
        auditor=FaceAuditor(device="cuda",torchcache=manifest["torchcache"])
    if auditor.metadata["state_sha256"]!=manifest["auditor_sha256"]: raise RuntimeError("Auditor changed")
    scored=0
    for seed in config["seeds"]:
        selected=[t for t in available if t["seed"]==seed]
        if not selected: continue
        ctx=_parallel_context(manifest,seed,auditor,make_uploads=False)
        target=config["target_participant"]
        for level,rep in sorted({(t["level"],t["realization"]) for t in selected}):
            same=[t for t in selected if t["level"]==level and t["realization"]==rep]
            B=noisy_anchor_bank(ctx["bank"],seed,level,rep,same[0]["scale"])
            for method in ("OP","MP","AM"):
                targets=same if method=="AM" else [same[0]]
                for task in targets:
                    p=task["p"] if method=="AM" else None
                    attack_id=f"s{seed}_{'p'+str(p) if p is not None else 'shared'}_l{level}_r{rep}_{method}"
                    path=attacks_dir/(attack_id+".json")
                    if path.exists():
                        old=json.loads(path.read_text())
                        if old.get("canonical_sha256")!=manifest["_canonical_sha256"] or old.get("execution_code_sha256")!=code_hash:
                            raise RuntimeError("Attack replay checkpoint configuration changed")
                        scored+=1; continue
                    diag=None
                    if method=="OP":
                        # Preserve the exact existing OP evaluation; do not rescore or refit it.
                        result=completed[task["run_id"]]["row"]["privacy"]
                        for peer in same:
                            other=completed[peer["run_id"]]["row"]["privacy"]
                            if any(other[key]!=result[key] for key in ("linkage_rate","per_query_success")):
                                raise RuntimeError("Fixed-target OP evaluations disagree across p")
                    else:
                        if method=="MP": qhat=_parallel_mp(ctx,B,target)
                        else:
                            cached=Path(parallel_root)/"jobs"/task["owner"]/"attack_inputs"/(task["run_id"]+".npz")
                            if cached.exists():
                                with np.load(cached,allow_pickle=False) as payload:
                                    if str(payload["canonical_sha256"])!=manifest["_canonical_sha256"] or str(payload["run_id"])!=task["run_id"]:
                                        raise RuntimeError("AM query artifact mismatch")
                                    qhat=payload["qhat"].copy()
                            else:
                                # Legacy rows have no maps: rerun alignment only; utility stays untouched.
                                maps,diag=align_anchors(B[:p],device="cuda",tol=config["gpm_tolerance"],max_iters=config["gpm_max_iters"])
                                qhat=_parallel_am(ctx,B,maps,target)
                                del maps
                            result=_parallel_privacy(ctx,auditor,qhat)
                        if method=="MP": result=_parallel_privacy(ctx,auditor,qhat)
                    _run_json(path,{"attack_id":attack_id,"protocol":"NAA-GDP","attack":method,"outer_seed":seed,
                                   "participant_count":p,"level_index":level,"realization":rep,"noise_scale":task["scale"],
                                   "privacy":result,"linkage_rate":result["linkage_rate"],"alignment_replayed":diag is not None,
                                   "alignment":diag,"canonical_sha256":manifest["_canonical_sha256"],
                                   "execution_code_sha256":code_hash,"source_utility_run":task["run_id"],"extra_utility_fits":0})
                    scored+=1
                    _run_json(root/"attack_progress.json",{"completed_attack_rows":scored,"available_NAA_utility_rows":len(available),
                              "expected_final_attack_rows":len(config["seeds"])*len(config["anchor_scales"])*config["realizations"]*7,
                              "updated_utc":datetime.now(timezone.utc).isoformat(),"current":attack_id})
        del ctx,B
        gc.collect(); torch.cuda.empty_cache()
    total_expected=len(config["seeds"])*len(config["anchor_scales"])*config["realizations"]*7
    count=sum(1 for _ in attacks_dir.glob("*.json"))
    _run_json(root/"attack_progress.json",{"status":"complete" if count==total_expected else "awaiting_peer_utility_results",
                "completed_attack_rows":count,"expected_final_attack_rows":total_expected,
                "available_NAA_utility_rows":len(available),"extra_utility_fits":0})
    return count


def merge_parallel_results(canonical_manifest_path,legacy_root,parallel_root):
    with _parallel_exclusive(Path(parallel_root)/"jobs"/"p5_10","merge.lock"):
        return _parallel_merge_impl(canonical_manifest_path,legacy_root,parallel_root)


def _parallel_merge_impl(canonical_manifest_path,legacy_root,parallel_root):
    """Single reporting owner p5_10; includes immutable legacy rows by reference."""
    manifest=_parallel_manifest(canonical_manifest_path)
    config=manifest["scientific_config"]
    tasks=[task for owner in PARALLEL_JOBS for task in parallel_schedule(config,owner)]
    completed=_parallel_collect(tasks,manifest,legacy_root,parallel_root)
    destination=Path(parallel_root)/"merged"
    raw=_parallel_report(destination,completed,tasks,config,len(completed)==len(tasks))
    fits=sum(t["fit_count"] for t in tasks if t["run_id"] in completed)
    statuses={owner:{"completed_fits":sum(t["fit_count"] for t in tasks if t["owner"]==owner and t["run_id"] in completed),
                     "scheduled_fits":sum(t["fit_count"] for t in tasks if t["owner"]==owner)} for owner in PARALLEL_JOBS}
    _run_json(destination/"progress.json",{"completed_fits":fits,"scheduled_fits":expected_fits(config),"jobs":statuses,
               "completed_main_result_rows":len(raw),"expected_main_result_rows":expected_result_rows(config),
               "utility_complete":len(completed)==len(tasks),"legacy_records_preserved":True})
    attack_paths=sorted((Path(parallel_root)/"jobs"/"p5_10"/"attacks").glob("*.json"))
    attacks=[json.loads(path.read_text()) for path in attack_paths]
    if attacks:
        pd.DataFrame([{k:v for k,v in row.items() if v is None or isinstance(v,(str,int,float,bool))} for row in attacks]).to_csv(destination/"attack_results.csv",index=False)
        joined=[]
        lookup={(a["outer_seed"],a["level_index"],a["realization"],a["participant_count"],a["attack"]):a for a in attacks}
        for _,row in raw[raw.method=="NAA-GDP"].iterrows():
            for method in ("OP","MP","AM"):
                key=(int(row.outer_seed),int(row.level_index),int(row.realization),int(row.participant_count) if method=="AM" else None,method)
                if key in lookup:
                    joined.append({"run_id":row.run_id,"method":"NAA-GDP","outer_seed":row.outer_seed,"participant_count":row.participant_count,
                                   "noise_scale":row.noise_scale,"realization":row.realization,"attack":method,
                                   "balanced_accuracy":row.balanced_accuracy,"linkage_rate":lookup[key]["linkage_rate"]})
        # The common-transform private baseline uses its disclosed transform;
        # displaying it in every panel does not relabel it as an MP/AM/OP attack.
        for _,row in raw[raw.method.isin(["C-GDP-private-noise","C-GDP","AA-GDP"])].iterrows():
            joined.append({"run_id":row.run_id,"method":row.method,"outer_seed":row.outer_seed,
                           "participant_count":row.participant_count,"noise_scale":row.noise_scale,
                           "realization":row.realization,
                           "attack":"known-common-transform" if row.method!="AA-GDP" else "noiseless-exact-recovery",
                           "balanced_accuracy":row.balanced_accuracy,"linkage_rate":row.linkage_rate})
        attack_long=pd.DataFrame(joined)
        attack_long.to_csv(destination/"privacy_utility_by_attack.csv",index=False)
        _parallel_attack_reporting(destination,attack_long,config,
                                   len(completed)==len(tasks) and len(attacks)==1400)
    (destination/"REPORT.md").write_text(
        f"# Parallel CelebA progress\n\nCompleted fitted models: {fits}/{expected_fits(config)}.\n"
        f"Main result rows: {len(raw)}/{expected_result_rows(config)}.\n"
        f"Post-hoc attack rows: {len(attacks)}/1400; these require no utility retraining.\n\n"
        "All completed legacy fits are reused without editing their files. Local models and aggregates belong to p50; "
        "private-noise runs and the OP/MP/AM replay belong to p5_10. Main p=10 rows are an exact filter of the merged results. "
        "Seed SD is computed after averaging four noise realizations within each deployment. "
        "Additional attack results are kept separate from the original OP utility records. "
        "attack_seed_means.csv first averages four realizations; attack_summary.csv reports their "
        "mean and SD across the five deployment seeds. Incomplete cells remain provisional. "
        "privacy_utility_three_attacks.pdf compares MP, AM, and OP; the known-common-transform "
        "private-noise baseline and noiseless references are shared across panels. "
        "Rerun the owner replay/merge cell after peer jobs finish if this report is provisional.\n")
    return raw


def _parallel_attack_reporting(destination,records,config,schedule_complete=False):
    """Strict realization/seed coverage, then three reconstruction panels at p=10."""
    keys=["method","participant_count","noise_scale","attack","outer_seed"]
    metrics=["balanced_accuracy","linkage_rate"]
    if records.empty:
        return pd.DataFrame()
    if records.duplicated(keys+["realization"]).any():
        raise RuntimeError("Duplicate realization in merged attack results")
    cells=[]
    for key,group in records.groupby(keys,dropna=False):
        row=dict(zip(keys,key))
        expected=set(range(config["realizations"])) if row["noise_scale"]>0 else {0}
        actual=set(map(int,group.realization))
        row.update(completed_realizations=len(group),required_realizations=len(expected),
                   seed_cell_complete=actual==expected and len(group)==len(expected))
        for metric in metrics: row[metric]=group[metric].mean()
        cells.append(row)
    seed_means=pd.DataFrame(cells)
    seed_means.to_csv(Path(destination)/"attack_seed_means.csv",index=False)
    summary=[]
    for key,group in seed_means[seed_means.seed_cell_complete].groupby(keys[:-1],dropna=False):
        row=dict(zip(keys[:-1],key))
        seeds=set(map(int,group.outer_seed))
        row.update(n_seeds=len(group),final=seeds==set(config["seeds"]) and len(group)==len(config["seeds"]))
        for metric in metrics:
            row[metric+"_mean"]=group[metric].mean()
            row[metric+"_seed_sd"]=group[metric].std(ddof=1) if len(group)>1 else np.nan
        summary.append(row)
    summary=pd.DataFrame(summary)
    summary.to_csv(Path(destination)/"attack_summary.csv",index=False)
    if summary.empty: return summary
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,3,figsize=(12.8,4.3),sharey=True)
    selected=summary[summary.participant_count==10]
    utility_path=Path(destination)/"summary.csv"
    utility=pd.read_csv(utility_path) if utility_path.exists() else pd.DataFrame()
    for method,ax in zip(("MP","AM","OP"),axes):
        for protocol,color in (("NAA-GDP","tab:blue"),("C-GDP-private-noise","tab:orange")):
            wanted=method if protocol=="NAA-GDP" else "known-common-transform"
            series=selected[(selected.method==protocol)&(selected.attack==wanted)].sort_values("noise_scale")
            if len(series):
                ax.errorbar(100*series.linkage_rate_mean,100*series.balanced_accuracy_mean,
                            xerr=100*series.linkage_rate_seed_sd.fillna(0),
                            yerr=100*series.balanced_accuracy_seed_sd.fillna(0),
                            marker="o",capsize=2,color=color,label=protocol)
        for protocol,marker,color in (("C-GDP","s","tab:red"),("AA-GDP","x","tab:purple")):
            reference=selected[selected.method==protocol]
            if len(reference):
                ax.scatter(100*reference.linkage_rate_mean,100*reference.balanced_accuracy_mean,
                           marker=marker,color=color,label=protocol+" (noiseless)")
        if len(utility):
            for protocol,style,color in (("Local-only","--","tab:green"),("I-GDP","-.","grey")):
                baseline=utility[(utility.method==protocol)&(utility.participant_count==10)]
                if len(baseline):
                    ax.axhline(100*baseline.balanced_accuracy_mean.iloc[0],linestyle=style,color=color,
                               label=protocol+" utility")
        ax.axvline(10,color="grey",linestyle=":")
        ax.set(title=method+" reconstruction",xlabel="Frozen-auditor identity linkage (%)")
    axes[0].set_ylabel("Smiling balanced accuracy (%)")
    handles,labels=axes[-1].get_legend_handles_labels()
    if handles: axes[-1].legend(handles,labels,fontsize=7)
    final=bool(schedule_complete and len(selected) and selected.final.all())
    fig.suptitle("CelebA, p=10"+("" if final else " (provisional)"))
    fig.tight_layout()
    fig.savefig(Path(destination)/"privacy_utility_three_attacks.pdf")
    plt.close(fig)
    return summary


## Verify shared preparation
If the p=50 preparation stage has not finished, this cell stops safely. Run it again after preparation completes.


In [ ]:
assert Path(CANONICAL_PATH).is_file(), "Run the preparation cell in CelebA_P50 first."
_parallel_manifest(CANONICAL_PATH, CONFIG)
print("Shared preparation verified.")


## Run this worker
Resume by running all cells again. Existing valid fits are skipped.


In [ ]:
print(protocol_self_test())
print(attack_self_test())
results = run_parallel_experiment(CONFIG, PARALLEL_CODE_SHA256, JOB_ID, CANONICAL_PATH, LEGACY_ROOT, PARALLEL_ROOT, replay_attacks=True)


## Replay and merge
Run this cell after all three workers finish if the first pass reported awaiting peers. It scores missing MP/AM/OP results without fitting any utility models. This notebook alone owns the combined report.


In [ ]:
replay_parallel_attacks(CANONICAL_PATH, LEGACY_ROOT, PARALLEL_ROOT, PARALLEL_CODE_SHA256)
merge_parallel_results(CANONICAL_PATH, LEGACY_ROOT, PARALLEL_ROOT)
